<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/04-pythonic-tools/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 4: Exercises

Twelve exercises: four Easy, five Medium and three Hard. Most are generators, decorators and context managers. A couple are classes.

As in Module 3, each check runs short scenarios, such as "feed it an infinite iterator and make sure it doesn't hang" or "make sure the exception still comes out of the `with` block".

**Run the setup cell first.**

Rules:

- Standard library only. `itertools`, `functools`, `collections` and `contextlib` are all fair game, but try writing things by hand first where the exercise says so.
- Where a function should be a generator, the check tests that it really is one.
- Be careful with infinite iterators. If a cell runs forever, you've probably called `list()` on one. Use the stop button and look for it.
- Solutions are in `03_solutions.ipynb`.

In [ ]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import copy
import math
import inspect
import itertools
import time


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, float) or isinstance(b, float):
        if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=70):
    text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def _first(iterator, limit=10):
    """Take at most `limit` items, so a buggy infinite generator fails instead of hanging."""
    return list(itertools.islice(iterator, limit))


def check_take(f):
    def stops_early():
        it = iter([1, 2, 3, 4])
        f(2, it)
        return next(it)

    _scenarios("take", [
        ("first 3 items of an infinite iterator", lambda: f(3, itertools.count()), [0, 1, 2],
         "you can't call list() on an infinite iterator. Stop after n items"),
        ("asking for more than there is", lambda: f(5, [1, 2]), [1, 2], None),
        ("n = 0", lambda: f(0, [1, 2]), [], None),
        ("doesn't consume more than n items", stops_early, 3,
         "after taking 2 items, the next item in the iterator should still be 3"),
        ("returns a list", lambda: type(f(2, "abc")).__name__, "list", None),
    ])


def check_running_total(f):
    _scenarios("running_total", [
        ("is a generator", lambda: inspect.isgenerator(f([1])), True, "use yield"),
        ("basic", lambda: list(f([1, 2, 3])), [1, 3, 6], None),
        ("floats", lambda: list(f([0.5, 0.25])), [0.5, 0.75], None),
        ("empty input", lambda: list(f([])), [], None),
        ("works on an infinite iterator", lambda: list(itertools.islice(f(itertools.count(1)), 4)), [1, 3, 6, 10],
         "don't convert the input to a list; loop over it directly"),
    ])


def check_count_calls(f):
    def counts():
        @f
        def add(a, b):
            return a + b
        add(1, 2)
        add(3, 4)
        add(5, b=6)
        return add.calls

    def result():
        @f
        def add(a, b):
            return a + b
        return add(2, b=3)

    def keeps_metadata():
        @f
        def scale(x):
            """Multiply by two."""
            return 2 * x
        return (scale.__name__, scale.__doc__)

    def separate():
        @f
        def a():
            pass

        @f
        def b():
            pass
        a(); a(); b()
        return (a.calls, b.calls)

    _scenarios("count_calls", [
        ("returns the original result", result, 5, "return whatever the wrapped function returns"),
        ("counts calls", counts, 3, "store the count as an attribute on the wrapper: wrapper.calls"),
        ("starts at 0", lambda: f(lambda: None).calls, 0, None),
        ("keeps the name and docstring", keeps_metadata, ("scale", "Multiply by two."), "use @functools.wraps(func)"),
        ("each decorated function has its own count", separate, (2, 1), None),
    ])


def check_Timer(C):
    def measures():
        with C() as t:
            time.sleep(0.02)
        return t.elapsed >= 0.019

    def enter_returns_self():
        timer = C()
        with timer as t:
            pass
        return t is timer

    def survives_error():
        t = C()
        try:
            with t:
                raise KeyError("boom")
        except KeyError:
            pass
        return t.elapsed is not None

    def propagates():
        with C():
            raise KeyError("boom")

    _scenarios("Timer", [
        ("elapsed is None before use", lambda: C().elapsed, None, None),
        ("`with ... as t` gives the Timer itself", enter_returns_self, True, "__enter__ should return self"),
        ("measures elapsed time in seconds", measures, True, "use time.perf_counter() in __enter__ and __exit__"),
        ("elapsed is set even if the block raises", survives_error, True, None),
        ("exceptions are not swallowed", propagates, Raises(KeyError), "__exit__ should return False"),
    ])


def check_batched(f):
    _scenarios("batched", [
        ("is a generator", lambda: inspect.isgenerator(f([1], 1)), True, "use yield"),
        ("basic", lambda: _first(f([1, 2, 3, 4, 5], 2)), [[1, 2], [3, 4], [5]],
         "if the first batch repeats forever, call iter() on the input once and keep using that iterator"),
        ("works on a generator", lambda: _first(f((x for x in range(5)), 2)), [[0, 1], [2, 3], [4]],
         "generators have no len() and can't be sliced. Collect items one at a time"),
        ("lazy on an infinite iterator", lambda: next(f(itertools.count(), 3)), [0, 1, 2], None),
        ("empty input", lambda: _first(f([], 3)), [], "don't yield an empty batch at the end"),
        ("exact multiple", lambda: _first(f([1, 2, 3, 4], 2)), [[1, 2], [3, 4]], "don't yield an empty batch at the end"),
        ("size 0", lambda: _first(f([1, 2], 0)), Raises(ValueError), None),
    ])


def check_sliding_window(f):
    _scenarios("sliding_window", [
        ("basic", lambda: list(f([1, 2, 3, 4], 2)), [(1, 2), (2, 3), (3, 4)], "yield tuples"),
        ("window of 3", lambda: list(f("abcd", 3)), [("a", "b", "c"), ("b", "c", "d")], None),
        ("window bigger than the input", lambda: list(f([1, 2], 3)), [], None),
        ("window of 1", lambda: list(f([1, 2], 1)), [(1,), (2,)], None),
        ("works on a generator", lambda: list(f((x * 10 for x in range(4)), 2)), [(0, 10), (10, 20), (20, 30)], None),
        ("lazy on an infinite iterator", lambda: list(itertools.islice(f(itertools.count(), 2), 3)), [(0, 1), (1, 2), (2, 3)], None),
        ("n = 0", lambda: list(f([1, 2], 0)), Raises(ValueError), None),
    ])


def check_parse_scores(f):
    lines = ["alice,0.9", "", "# comment line", "  bob , 0.75 ", "broken line", "carol,abc", "dave,1", "eve,0.5,extra"]
    _scenarios("parse_scores", [
        ("is a generator", lambda: inspect.isgenerator(f([])), True, None),
        ("parses, cleans and skips bad lines", lambda: list(f(lines)),
         [("alice", 0.9), ("bob", 0.75), ("dave", 1.0)],
         "skip blank lines, '#' comments, and lines that don't have exactly a name and a number"),
        ("works lazily on a generator", lambda: next(f(line for line in ["x,1", "y,2"])), ("x", 1.0), None),
    ])


def check_retry(f):
    def flaky(fail_times, exc=ConnectionError):
        state = {"calls": 0}

        def call(x):
            state["calls"] += 1
            if state["calls"] <= fail_times:
                raise exc("temporary failure")
            return x * 2
        return call, state

    def succeeds():
        fn, state = flaky(2)
        wrapped = f(times=3)(fn)
        return (wrapped(5), state["calls"])

    def gives_up():
        fn, _ = flaky(5)
        f(times=3)(fn)(1)

    def gives_up_count():
        fn, state = flaky(5)
        try:
            f(times=3)(fn)(1)
        except ConnectionError:
            pass
        return state["calls"]

    def other_errors():
        fn, state = flaky(1, exc=ValueError)
        try:
            f(times=3, exceptions=(ConnectionError,))(fn)(1)
        except ValueError:
            pass
        return state["calls"]

    def keeps_name():
        @f(times=2)
        def download():
            """Fetch a file."""
        return (download.__name__, download.__doc__)

    _scenarios("retry", [
        ("succeeds after 2 failures", succeeds, (10, 3), None),
        ("raises the last error after `times` attempts", gives_up, Raises(ConnectionError), "re-raise the error after the last attempt"),
        ("makes exactly `times` attempts", gives_up_count, 3, None),
        ("doesn't retry other exception types", other_errors, 1, "only catch the exception types in `exceptions`"),
        ("keeps the name and docstring", keeps_name, ("download", "Fetch a file."), "use @functools.wraps"),
        ("times=0 is invalid", lambda: f(times=0), Raises(ValueError), "check `times` when retry() is called, before decorating anything"),
    ])


def check_memoize(f):
    def caches():
        calls = []

        @f
        def square(x):
            calls.append(x)
            return x * x
        results = [square(3), square(3), square(4), square(3)]
        return (results, calls)

    def recursive():
        calls = [0]

        @f
        def fib(n):
            calls[0] += 1
            return n if n < 2 else fib(n - 1) + fib(n - 2)
        return (fib(25), calls[0])

    def exposes_cache():
        @f
        def add(a, b):
            return a + b
        add(1, 2)
        add(3, 4)
        return add.cache

    def keeps_name():
        @f
        def embed(word):
            """Look up an embedding."""
        return embed.__name__

    _scenarios("memoize", [
        ("computes each argument only once", caches, ([9, 9, 16, 9], [3, 4]),
         "create the cache once, outside wrapper, so it survives between calls"),
        ("speeds up recursion", recursive, (75025, 26), "the recursive calls go through the decorated name, so they're cached too"),
        ("exposes the cache as a dict keyed by the args tuple", exposes_cache, {(1, 2): 3, (3, 4): 7}, None),
        ("keeps the name", keeps_name, "embed", "use @functools.wraps"),
    ])


def check_Registry(C):
    class Linear:
        def __init__(self, w=1.0):
            self.w = w

    def setup():
        r = C()

        @r.register("linear")
        class Lin(Linear):
            pass

        @r.register("relu")
        def relu(x):
            return max(0.0, x)
        return r, Lin, relu

    def returns_unchanged():
        r = C()

        def fn(x):
            return x + 1
        return r.register("inc")(fn) is fn

    def duplicate():
        r = C()
        r.register("a")(lambda: 1)
        r.register("a")(lambda: 2)

    _scenarios("Registry", [
        ("register returns the object unchanged", returns_unchanged, True, "the inner decorator should `return obj`"),
        ("membership", lambda: ("linear" in setup()[0], "conv" in setup()[0]), (True, False), "implement __contains__"),
        ("get", lambda: setup()[0].get("relu")(-3.0), 0.0, None),
        ("build calls the object with keyword arguments", lambda: setup()[0].build("linear", w=2.5).w, 2.5, None),
        ("names() is sorted", lambda: setup()[0].names(), ["linear", "relu"], None),
        ("len", lambda: len(setup()[0]), 2, None),
        ("registering a name twice", duplicate, Raises(ValueError), None),
        ("building an unknown name", lambda: setup()[0].build("conv"), Raises(KeyError), None),
    ])


def check_interleave(f):
    _scenarios("interleave", [
        ("two iterables", lambda: list(f([1, 2, 3], "ab")), [1, "a", 2, "b", 3],
         "zip() stops at the shortest input, so it loses the 3"),
        ("three iterables", lambda: list(f("ad", "be", "cf")), ["a", "b", "c", "d", "e", "f"], None),
        ("uneven lengths", lambda: list(f([1], [2, 3, 4], [5, 6])), [1, 2, 5, 3, 6, 4],
         "when one runs out, keep going with the others"),
        ("no iterables", lambda: list(f()), [], None),
        ("an empty one", lambda: list(f([], [1, 2])), [1, 2], None),
        ("lazy with an infinite iterator", lambda: list(itertools.islice(f(itertools.count(), "xy"), 6)),
         [0, "x", 1, "y", 2, 3], "don't convert the inputs to lists"),
    ])


def check_temporary_attrs(f):
    class Model:
        pass

    def make():
        m = Model()
        m.training = True
        m.dropout = 0.5
        return m

    def inside_and_after():
        m = make()
        with f(m, training=False, dropout=0.0):
            inside = (m.training, m.dropout)
        return (inside, (m.training, m.dropout))

    def new_attribute():
        m = make()
        with f(m, debug=True):
            inside = m.debug
        return (inside, hasattr(m, "debug"))

    def restores_on_error():
        m = make()
        try:
            with f(m, training=False):
                raise RuntimeError("boom")
        except RuntimeError:
            pass
        return m.training

    def propagates():
        with f(make(), training=False):
            raise RuntimeError("boom")

    def nested():
        m = make()
        with f(m, training=False):
            with f(m, training="eval"):
                pass
            middle = m.training
        return (middle, m.training)

    _scenarios("temporary_attrs", [
        ("sets attributes inside, restores them after", inside_and_after, ((False, 0.0), (True, 0.5)), None),
        ("a new attribute is removed afterwards", new_attribute, (True, False),
         "if the attribute didn't exist before, delete it on exit (delattr)"),
        ("restores even if the block raises", restores_on_error, True, "use try/finally around the yield"),
        ("doesn't swallow the exception", propagates, Raises(RuntimeError), None),
        ("nesting works", nested, (False, True), None),
    ])


print("Setup done. Checks are ready.")

## Easy

### Exercise 1: `take`

Write `take(n, iterable)` that returns a **list** of the first `n` items from any iterable, including infinite ones like `itertools.count()`.

```python
take(3, itertools.count())   # [0, 1, 2]
take(5, [1, 2])              # [1, 2]
```

It must not consume more than `n` items. If you pass in an iterator, the item after the ones you took should still be there afterwards.

In [ ]:
def take(n: int, iterable) -> list:
    # your code here
    pass

In [ ]:
check_take(take)

### Exercise 2: `running_total`

Write a **generator** `running_total(numbers)` that yields the cumulative sum after each number.

```python
list(running_total([1, 2, 3]))   # [1, 3, 6]
```

It must work on infinite inputs, so don't turn the input into a list.

In [ ]:
def running_total(numbers):
    # your code here
    pass

In [ ]:
check_running_total(running_total)

### Exercise 3: `count_calls`

Write a decorator `count_calls` that counts how many times a function is called.

- The decorated function should behave exactly like the original, returning the same result for the same arguments (positional and keyword).
- The count is stored in an attribute `.calls` on the decorated function, starting at 0.
- The decorated function keeps its original `__name__` and `__doc__`.

```python
@count_calls
def add(a, b):
    return a + b

add(1, 2); add(3, 4)
add.calls   # 2
```

In [ ]:
import functools


def count_calls(func):
    # your code here
    pass

In [ ]:
check_count_calls(count_calls)

### Exercise 4: `Timer`

Write a context manager class `Timer`:

- `elapsed` is `None` until a `with` block has finished.
- `with Timer() as t:` gives you the `Timer` object itself.
- When the block ends, `t.elapsed` holds the time it took, in seconds, measured with `time.perf_counter()`.
- `elapsed` must be set even if the block raises an exception, and the exception must not be swallowed.

```python
with Timer() as t:
    train_one_epoch()
print(f"epoch took {t.elapsed:.2f} s")
```

In [ ]:
import time


class Timer:
    # your code here
    pass

In [ ]:
check_Timer(Timer)

## Medium

### Exercise 5: `batched`

Write a **generator** `batched(iterable, size)` that yields lists of `size` consecutive items. The last batch can be shorter.

```python
list(batched([1, 2, 3, 4, 5], 2))   # [[1, 2], [3, 4], [5]]
```

- It must work on any iterable, including generators, which have no `len()` and can't be sliced.
- It must be lazy: on an infinite iterator, the first batch should come out right away.
- Never yield an empty batch.
- Raise a `ValueError` if `size` is less than 1.

In [ ]:
def batched(iterable, size: int):
    # your code here
    pass

In [ ]:
check_batched(batched)

### Exercise 6: `sliding_window`

Write a **generator** `sliding_window(iterable, n)` that yields every run of `n` consecutive items as a tuple.

```python
list(sliding_window([1, 2, 3, 4], 2))   # [(1, 2), (2, 3), (3, 4)]
```

- If there are fewer than `n` items, yield nothing.
- It must work on generators and infinite iterators.
- Raise a `ValueError` if `n` is less than 1.

Sliding windows are everywhere in time-series ML: to predict the next value from the last `n`, you train on windows of `n` values.

Hint: a `deque` with `maxlen=n` from Module 2 does most of the work.

In [ ]:
from collections import deque


def sliding_window(iterable, n: int):
    # your code here
    pass

In [ ]:
check_sliding_window(sliding_window)

### Exercise 7: `parse_scores`

Write a **generator** `parse_scores(lines)` that reads lines like `"alice,0.9"` and yields `(name, score)` tuples, with the score as a float.

Skip:

- blank lines (including lines that are only spaces)
- lines starting with `#`
- lines that don't have exactly two comma-separated fields
- lines where the score isn't a number

Strip spaces around each field, so `"  bob , 0.75 "` gives `("bob", 0.75)`.

```python
lines = ["alice,0.9", "", "# comment", "carol,abc", "dave,1"]
list(parse_scores(lines))   # [("alice", 0.9), ("dave", 1.0)]
```

In [ ]:
def parse_scores(lines):
    # your code here
    pass

In [ ]:
check_parse_scores(parse_scores)

### Exercise 8: `retry`

Write a decorator factory `retry(times, exceptions=(Exception,))`:

- `@retry(times=3)` makes the decorated function try up to 3 times in total.
- Only exceptions of the types in `exceptions` trigger a retry. Any other exception is raised immediately.
- If every attempt fails, raise the last exception.
- Keep the original function's name and docstring.
- Raise a `ValueError` straight away, when `retry(...)` is called, if `times` is less than 1.

```python
@retry(times=3, exceptions=(ConnectionError,))
def download(url):
    ...
```

In [ ]:
import functools


def retry(times: int, exceptions=(Exception,)):
    # your code here
    pass

In [ ]:
check_retry(retry)

### Exercise 9: `memoize`

Write your own version of `functools.cache`: a decorator `memoize` that stores results by arguments.

- The first call with some arguments runs the function and stores the result. Later calls with the same arguments return the stored result without running the function.
- Only positional arguments need to be supported.
- Expose the cache as a dict attribute `.cache` on the decorated function, keyed by the tuple of arguments.
- Keep the original function's name.

```python
@memoize
def square(x):
    return x * x

square(3); square(3); square(4)
square.cache   # {(3,): 9, (4,): 16}
```

In [ ]:
import functools


def memoize(func):
    # your code here
    pass

In [ ]:
check_memoize(memoize)

## Hard

### Exercise 10: `Registry`

Write a `Registry` class for registering models (or functions) by name, the pattern used by libraries like timm and Hugging Face.

- `register(name)` returns a decorator. Applying it to a class or function records it under `name` and returns it **unchanged**. Registering the same name twice raises a `ValueError`.
- `get(name)` returns the registered object.
- `build(name, **kwargs)` calls the registered object with the keyword arguments and returns the result. For a class, that creates an instance. Raise a `KeyError` for an unknown name.
- `name in registry` works, and so does `len(registry)`.
- `names()` returns a sorted list of registered names.

```python
MODELS = Registry()

@MODELS.register("linear")
class LinearModel:
    def __init__(self, w=1.0):
        self.w = w

model = MODELS.build("linear", w=2.0)
```

In [ ]:
class Registry:
    # your code here
    pass

In [ ]:
check_Registry(Registry)

### Exercise 11: `interleave`

Write a **generator** `interleave(*iterables)` that takes items from each iterable in turn, round-robin style. When one runs out, keep going with the rest until all are exhausted.

```python
list(interleave([1, 2, 3], "ab"))      # [1, "a", 2, "b", 3]
list(interleave([1], [2, 3, 4], [5, 6]))   # [1, 2, 5, 3, 6, 4]
```

It must work with infinite iterators, so don't convert anything to a list.

This is how multi-task or multi-dataset training often mixes data: one batch from each source in turn.

In [ ]:
def interleave(*iterables):
    # your code here
    pass

In [ ]:
check_interleave(interleave)

### Exercise 12: `temporary_attrs`

Write a context manager `temporary_attrs(obj, **attrs)` that temporarily sets attributes on an object, then puts everything back exactly as it was.

- Inside the `with` block, each attribute has its new value.
- Afterwards, attributes that existed before get their old values back, and attributes that didn't exist before are deleted.
- Everything is restored even if the block raises an exception, and the exception isn't swallowed.
- Nested uses must work.

```python
with temporary_attrs(model, training=False, dropout=0.0):
    evaluate(model)
# model.training and model.dropout are back to their old values
```

This is the pattern behind `model.eval()` and `torch.no_grad()`. Use `@contextmanager`, and the built-ins `hasattr`, `getattr`, `setattr` and `delattr`.

In [ ]:
from contextlib import contextmanager


def temporary_attrs(obj, **attrs):
    # your code here
    pass

In [ ]:
check_temporary_attrs(temporary_attrs)

## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then head to the mini project in `04_mini_project.ipynb`, where you'll build a small hyperparameter search framework using generators, decorators, a registry and context managers.